# GNN Comparison on QM9 — Training Notebook
**Models:** GCN · GAT · SchNet  
**Dataset:** QM9 (Ramakrishnan et al., 2014)  
**Task:** Molecular property regression

---
**Instructions:**
1. Mount Google Drive (Cell 2) — logs/checkpoints save there
2. Install dependencies (Cell 3)
3. Clone/upload your project repo (Cell 4)
4. Configure your run (Cell 5)
5. Run the model cell you want (GCN / GAT / SchNet)
6. Compare results in the final cell

In [ ]:
# ── Cell 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# All outputs (logs, checkpoints, results) will be saved here
DRIVE_ROOT = '/content/drive/MyDrive/gnn_qm9'

import os
os.makedirs(f'{DRIVE_ROOT}/outputs/logs',        exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/outputs/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/outputs/results',     exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data',                exist_ok=True)
print('Drive mounted. Output root:', DRIVE_ROOT)

In [ ]:
# ── Cell 2: Install Dependencies ─────────────────────────────────────────────
import subprocess, sys

# Check torch version first — PyG wheels are version-specific
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

TORCH_VERSION = torch.__version__.split('+')[0]   # e.g. '2.1.0'
CUDA_VERSION  = 'cu121' if torch.cuda.is_available() else 'cpu'

# Install PyTorch Geometric and dependencies
!pip install -q torch-geometric
!pip install -q pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html

# Other deps
!pip install -q pyyaml tqdm

print('\nInstallation complete.')

In [ ]:
# ── Cell 3: Clone Project Repo ───────────────────────────────────────────────
# Option A: Clone from GitHub (recommended for team)
# !git clone https://github.com/YOUR_USERNAME/gnn_qm9.git /content/gnn_qm9

# Option B: Upload a zip and unzip
# from google.colab import files
# uploaded = files.upload()  # upload gnn_qm9.zip
# !unzip gnn_qm9.zip -d /content/

# Option C: If already cloned
PROJECT_DIR = '/content/gnn_qm9'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Project files:', os.listdir('.'))

In [ ]:
# ── Cell 4: Configuration ─────────────────────────────────────────────────────
# Edit this cell to change what you're running.
# All models in the same session must use the SAME config for fair comparison.

CONFIG = {
    'dataset': {
        'target': 7,              # QM9 target index
                                  # 0=mu, 1=alpha, 2=eps_HOMO, 3=eps_LUMO,
                                  # 4=delta_eps, 5=R2, 6=ZPVE, 7=U0 (default)
                                  # 8=U, 9=H, 10=G, 11=Cv
        'split': [0.8, 0.1, 0.1],
        'seed': 42,
        'feature_mode': 'topology',  # GCN/GAT use 'topology'; SchNet uses 'geometry'
    },
    'training': {
        'epochs': 200,
        'batch_size': 32,
        'lr': 1e-3,
        'scheduler': 'cosine',    # 'cosine' | 'step' | 'none'
        'patience': 20,           # early stopping
    },
    'model': {
        'hidden_dim': 128,
        'num_layers': 4,
        'dropout': 0.0,
    }
}

In [ ]:
# ── Cell 5: Load Data ─────────────────────────────────────────────────────────
# Run once — shared across all model experiments
from data.loader import get_dataloaders


# Data root: save QM9 to Drive so it doesn't re-download every session
DATA_ROOT = f'{DRIVE_ROOT}/data/qm9_raw'

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Config:', CONFIG)


train_loader, val_loader, test_loader, normalizer = get_dataloaders(
    CONFIG, root=DATA_ROOT
)

# Sanity check — inspect one batch
batch = next(iter(train_loader))
print('\nBatch fields:', batch.keys)
print('x shape:         ', batch.x.shape)         # [N_total, node_dim]
print('edge_index shape:', batch.edge_index.shape) # [2, E_total]
print('y shape:         ', batch.y.shape)          # [B] or [B, 1]
print('pos shape:       ', batch.pos.shape)        # [N_total, 3]
print('z shape:         ', batch.z.shape)          # [N_total] atomic numbers
print('num_graphs:      ', batch.num_graphs)

In [ ]:
# ── Cell 6: Train GCN ─────────────────────────────────────────────────────────
from train import train_model

OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'

gcn_result = train_model(
    CONFIG,
    device=str(DEVICE),
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    model_name='gcn',
)

print(f"\nBest validation MAE: {gcn_result['best_val_mae']:.6f}")
print(f"Checkpoint: {gcn_result['checkpoint']}")

In [ ]:
# ── Cell 7: Evaluate on Test Set ──────────────────────────────────────────────
from evaluate import evaluate_model

test_result = evaluate_model(
    CONFIG,
    checkpoint_path=gcn_result['checkpoint'],
    device=str(DEVICE),
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    model_name='gcn',
)

print(f"\nTest MAE:  {test_result['test_mae']:.6f}")
print(f"Test RMSE: {test_result['test_rmse']:.6f}")

In [ ]:
# ── Cell 8: Training Curves ───────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

history = pd.read_csv(f'{OUTPUT_DIR}/logs/gcn_history.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['epoch'], history['train_loss'], label='Train Loss')
axes[0].plot(history['epoch'], history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss (normalized)')
axes[0].set_title('GCN — Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['epoch'], history['val_mae'], label='Val MAE', color='tab:orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (denormalized)')
axes[1].set_title('GCN — Validation MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/results/gcn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {OUTPUT_DIR}/results/gcn_training_curves.png")